In [ ]:
# # Only use when connecting to Colab GPU from VSCode extension
# # Sets up Colab environment --> Should take max ~3 minutes

# from google.colab import drive
# import os
# drive.mount('/content/drive')

# !mkdir -p ~/.ssh
# !cp /content/drive/MyDrive/colab_keys/id_ed25519 ~/.ssh/
# !cp /content/drive/MyDrive/colab_keys/id_ed25519.pub ~/.ssh/

# !chmod 600 ~/.ssh/id_ed25519
# !ssh-keyscan github.com >> ~/.ssh/known_hosts

# if not os.path.exists("CMPM118-Winter2026"):
#     !git clone git@github.com:Tardigrades3/CMPM118-Winter2026.git
    
# %cd CMPM118-Winter2026
# !git checkout aran-dev
# !python download_ninapro.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
/content/CMPM118-Winter2026
Already on 'aran-dev'
Your branch is up to date with 'origin/aran-dev'.
'NinaProData' already exists. Skipping download.


In [4]:
import torch
from torch import nn
from tqdm import tqdm
import copy
from torch.utils.data import Subset
import torch.nn.functional as F

import sys, os
sys.path.append(os.path.abspath("../"))
import preprocessing
import numpy as np



In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# Base MNIST NN

class NinaProClassificationModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv1d(10, 128, kernel_size=5, stride=2)
    self.relu1 = nn.ReLU()
    self.pool1 = nn.MaxPool1d(kernel_size=2)
    self.drop1 = nn.Dropout(0.1)

    self.flatten = nn.Flatten()
    
    self.dense1 = nn.LazyLinear(256)
    self.relu3 = nn.ReLU()
    self.drop3 = nn.Dropout(0.4)
    self.dense2 = nn.Linear(256, 13)

  def forward(self, x):
    out = self.relu1(self.conv1(x))
    out = self.pool1(out)
    out = self.drop1(out)
    out = self.flatten(out)
    out = self.relu3(self.dense1(out))
    out = self.drop3(out)
    out = self.dense2(out)
    return out

class NinaProDataset(torch.utils.data.Dataset):
  def __init__(self, x, y):
    self.X = torch.tensor(x, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.long)

  def __len__(self):
    return len(self.X)
  
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

# Base NN


model = NinaProClassificationModel().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# path should be ../NinaProData for local machine
x_train, y_train, x_test, y_test, class_weights = preprocessing.multi_preprocess(exercise_number=1, path="../NinaProData")
batch_size = 100

# Since resting is a massive portion of our dataset, remove all resting labels for now.
def drop_rest_label(x_train, y_train, x_test, y_test):
    mask = y_train != 0
    x_train = x_train[mask]
    y_train = y_train[mask]

    mask = y_test != 0
    x_test = x_test[mask]
    y_test = y_test[mask]

    y_train -= 1
    y_test -= 1
    return x_train, y_train, x_test, y_test

x_train, y_train, x_test, y_test = drop_rest_label(x_train, y_train, x_test, y_test)

train_data = NinaProDataset(x_train, y_train)
test_data = NinaProDataset(x_test, y_test)
batched_train = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
batched_test = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False)


loss_func = nn.CrossEntropyLoss()

Using cuda device
subject #[[1]]
exercise #[[1]]
0.0     351
5.0      58
3.0      53
1.0      50
8.0      44
6.0      43
12.0     42
4.0      41
10.0     39
2.0      37
7.0      36
9.0      35
11.0     35
Name: count, dtype: int64
0.0     908
3.0      30
1.0      26
10.0     25
12.0     23
5.0      22
4.0      18
6.0      17
7.0      17
9.0      17
2.0      16
8.0      16
11.0     15
Name: count, dtype: int64
subject #[[2]]
exercise #[[1]]
0.0     291
5.0      55
7.0      54
3.0      53
8.0      50
11.0     50
4.0      49
2.0      48
9.0      48
6.0      46
12.0     42
10.0     39
1.0      37
Name: count, dtype: int64
0.0     880
8.0      28
3.0      26
1.0      24
5.0      24
4.0      23
7.0      22
9.0      21
10.0     21
2.0      20
6.0      20
11.0     18
12.0     18
Name: count, dtype: int64
subject #[[3]]
exercise #[[1]]
0.0     339
2.0      54
3.0      52
5.0      47
6.0      47
12.0     46
1.0      44
7.0      43
4.0      42
8.0      40
9.0      39
10.0     35
11.0     34
Name:

In [7]:
# TODO: Ideas to improve model
# Filter out resting position[Done - didn't work, only exposed overfitting problem]
# Integrate class weights somehow to fix imbalances

print("Starting Training")
epochs = 20
model.train()

for epoch in range(epochs):
    pbar = tqdm(batched_train, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    for emg, label in pbar:
        emg = emg.to(device)
        label = label.to(device)
        optimizer.zero_grad()
        logits = model(emg)
        loss = loss_func(logits, label)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item())
        

Starting Training


In [8]:
# Accuracy:
def find_accuracy(model, batched_test):
    model.eval()
    total_loss = 0
    total = 0
    correct = 0
    with torch.no_grad():
        correct = 0
        for emg, labels in batched_test:
            emg = emg.to(device)
            labels = labels.to(device)
            logits = model(emg)
            loss = loss_func(logits, labels)

            total_loss += loss.item() * emg.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += emg.size(0)
    print(f"Average Loss: {total_loss/correct}, Accuracy: {correct/total}")
find_accuracy(model, batched_test)

Average Loss: 1.7552880971215312, Accuracy: 0.6810576063354196


In [10]:
from torchmetrics.classification import MulticlassAccuracy

num_classes = 12
per_class_acc = MulticlassAccuracy(
    num_classes=num_classes,
    average=None
).to(device)

model.eval()

with torch.no_grad():
    for emg, label in batched_train:
        emg = emg.to(device).float()
        label = label.to(device)
        logits = model(emg)
        preds = logits.argmax(dim=1)
        per_class_acc.update(preds, label)

accs = per_class_acc.compute()


In [11]:
for i, a in enumerate(accs):
    print(f"Class {i}: {a.item():.3f}")

Class 0: 0.936
Class 1: 0.957
Class 2: 0.927
Class 3: 0.954
Class 4: 0.957
Class 5: 0.892
Class 6: 0.951
Class 7: 0.947
Class 8: 0.838
Class 9: 0.853
Class 10: 0.860
Class 11: 0.920


In [12]:
def expand_classifier(model, new_size, device):
  old_fc = model.dense2
  in_features = old_fc.in_features
  old_out = old_fc.out_features
  newlayer = nn.Linear(in_features, old_out + new_size).to(device)

  with torch.no_grad():
    newlayer.weight[:old_out].copy_(old_fc.weight)
    newlayer.bias[:old_out].copy_(old_fc.bias)
  model.dense2 = newlayer
  return model

In [14]:
# CIL / KWF

x2_train, y2_train, x2_test, y2_test, class_weights_2 = preprocessing.multi_preprocess(2, "../NinaProData")

# Since resting is a massive portion of our dataset, remove all resting labels for now.
x2_train, y2_train, x2_test, y2_test = drop_rest_label(x2_train, y2_train, x2_test, y2_test)


train2_data = NinaProDataset(x2_train, y2_train)
test2_data = NinaProDataset(x2_test, y2_test)

batch_size = 100
train2_loader = torch.utils.data.DataLoader(train2_data, batch_size=batch_size, shuffle=True)
test2_loader = torch.utils.data.DataLoader(test2_data, batch_size=batch_size, shuffle=False)

num_new_classes = len(np.unique(y2_train))

student = copy.deepcopy(model).to(device)
student = expand_classifier(student, new_size=num_new_classes, device=device)
student = student.to(device)



subject #[[1]]
exercise #[[2]]
0.0     488
17.0     59
9.0      50
12.0     50
1.0      49
6.0      47
16.0     47
13.0     46
2.0      45
4.0      45
7.0      43
3.0      42
10.0     41
8.0      39
5.0      36
11.0     36
15.0     34
14.0     30
Name: count, dtype: int64
0.0     1300
6.0       26
17.0      26
3.0       22
12.0      22
16.0      21
10.0      20
2.0       19
7.0       19
8.0       19
9.0       19
11.0      19
15.0      19
1.0       18
13.0      17
5.0       15
4.0       14
14.0      11
Name: count, dtype: int64
subject #[[2]]
exercise #[[2]]
0.0     357
1.0      63
17.0     59
11.0     57
16.0     56
2.0      55
4.0      55
10.0     54
7.0      53
3.0      52
9.0      50
13.0     49
5.0      47
6.0      46
12.0     45
14.0     44
8.0      42
15.0     42
Name: count, dtype: int64
0.0     1259
1.0       27
11.0      25
17.0      25
4.0       23
9.0       23
13.0      23
16.0      23
2.0       22
10.0      21
14.0      21
3.0       20
5.0       20
7.0       20
6.0       18

In [16]:
student = student.to(device)
student.train()

optimizer = torch.optim.Adam(student.parameters(), lr = 0.001)
epochs = 100
T = 4
alpha_kd = .9

for epoch in range(epochs):
    # if epoch%10 == 0 or epoch < 5: find_accuracy(student, test2_loader)
    pbar = tqdm(train2_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    for emg, labels in pbar:
        emg = emg.to(device)
        labels = labels.to(device)
        logits_stud_new = student(emg)
        loss_stud_new = loss_func(logits_stud_new[:, 13:], labels)
        
        with torch.no_grad():
            logits_teach = model(emg)
            loss_teach = F.softmax(logits_teach/T, dim = 1)
        loss_stud_old = F.log_softmax(logits_stud_new[:, :13]/T, dim=1)
        loss_old = F.kl_div(loss_stud_old, loss_teach, reduction = "batchmean") * (T**2)
        
        loss = (1-alpha_kd) * loss_stud_new + alpha_kd * loss_old
        optimizer.zero_grad()
                
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item())
        


In [17]:
print(f"Accuracy for old task")
find_accuracy(student, batched_test)
print(f"Accuracy for new task")
find_accuracy(student, test2_loader)

Accuracy for old task
Average Loss: 3.4018867954559857, Accuracy: 0.5132200791927449
Accuracy for new task
Average Loss: 245.87044597910597, Accuracy: 0.06085753803596127
